<a href="https://colab.research.google.com/github/bsheese/cs377/blob/main/06_pandas_intro/06_6_Pivot_Crosstab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 06.6 — Pivot Tables, Crosstabs, and Descriptive Tools

## Learning Objectives

By the end of this notebook you should be able to:

- Use `pd.crosstab()` to produce a frequency or proportion table for two categorical variables
- Normalize a crosstab to answer "what fraction?" instead of "how many?"
- Use `pd.pivot_table()` to aggregate any numeric column across two dimensions simultaneously
- Explain when to reach for groupby vs crosstab vs pivot_table
- Interpret a correlation matrix from `.corr()`
- Use `.nlargest()` and `.nsmallest()` to find outliers

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/bsheese/CSDS125ExampleData/master/data_titanic.csv"
df = pd.read_csv(url)
df.columns = df.columns.str.lower()
df = df.rename(columns={
    'siblings/spouses aboard': 'sibsp',
    'parents/children aboard': 'parch'
})
df.head()

## The Gap GroupBy Leaves

By now you can answer questions like "what is the average survival rate per class?" with `.groupby("pclass")["survived"].mean()`. That gives you one number per class — a single column of results.

But some questions are inherently two-dimensional. *"How does survival rate vary across both class AND sex simultaneously?"* The answer is a grid: three rows (one per class) and two columns (one per sex), with a survival rate in each cell. That grid is what a **pivot table** and a **crosstab** produce.

The goal of this notebook is to build that 2-D view of the Titanic data, and to introduce a few other descriptive tools for understanding a dataset at a glance.

## 1. `pd.crosstab()` — Counts Across Two Categories

`pd.crosstab()` answers the question: *"how many passengers fall into each combination of two categories?"*

Let's start with the most basic question: how many men and women survived vs did not?

In [ ]:
pd.crosstab(df["sex"], df["survived"])

The table tells you: 233 women survived and 81 did not; 109 men survived and 468 did not. But raw counts make it hard to compare because there were more men than women aboard. What you really want is *proportions*.

### From Counts to Proportions

The `normalize` parameter converts counts to fractions. The key question is: fractions *of what?*

- `normalize="index"` — each **row** sums to 1. *"Of all women, what fraction survived?"*
- `normalize="columns"` — each **column** sums to 1. *"Of all survivors, what fraction were women?"*
- `normalize="all"` — the whole table sums to 1. *"What fraction of all passengers were women who survived?"*

In [ ]:
# Of all passengers in each sex, what fraction survived?
pd.crosstab(df["sex"], df["survived"], normalize="index").round(2)

Now the story is clear: 74% of women survived, 19% of men. That's the number you want when the question is "which group had better odds?"

In [ ]:
# Of all survivors, what fraction were women?
# normalize='columns' is useful when the question is about the *composition* of a group
pd.crosstab(df["sex"], df["survived"], normalize="columns").round(2)

In [ ]:
# Add row and column totals with margins=True
pd.crosstab(
    df["pclass"], df["survived"],
    rownames=["Class"],
    colnames=["Survived"],
    normalize="index",
    margins=True,
).round(2)

The margins row (`All`) shows the overall survival rate across all classes. The table immediately shows that first-class survival (63%) was more than twice that of third-class (24%).

## 2. `pd.pivot_table()` — Any Aggregation in a 2-D Grid

`crosstab` is convenient for *counting and proportions*. But what if you want the *average fare* broken down by class and sex? Or the *average age*? For any aggregation beyond counting, use `pd.pivot_table()`.

You can think of `pivot_table` as a `.groupby()` whose output is automatically arranged into a 2-D grid instead of a flat list.

In [ ]:
# What was the average survival rate for each class-sex combination?
pd.pivot_table(
    df,
    values  = "survived",
    index   = "pclass",
    columns = "sex",
    aggfunc = "mean",
).round(2)

This is the most important table in the Titanic dataset. Read across any row: women always outsurvived men. Read down any column: higher class means higher survival for both sexes. But third-class women (49%) still outsurvived first-class men (37%) — being female mattered more than being wealthy.

In [ ]:
# What was the average fare by class and sex?
pd.pivot_table(
    df,
    values  = "fare",
    index   = "pclass",
    columns = "sex",
    aggfunc = "mean",
).round(2)

In [ ]:
# Add overall row/column averages with margins=True
pd.pivot_table(
    df,
    values       = "survived",
    index        = "pclass",
    columns      = "sex",
    aggfunc      = "mean",
    margins      = True,
    margins_name = "Overall",
).round(2)

### Multiple Aggregations at Once

In [ ]:
# Both the count and the mean — useful to see sample size alongside rate
pd.pivot_table(
    df,
    values  = "fare",
    index   = "pclass",
    columns = "sex",
    aggfunc = ["mean", "count"],
).round(2)

## 3. Choosing the Right Tool

All three tools summarize grouped data. Here is how to decide which to reach for:

| Tool | Reach for it when… |
|---|---|
| `.groupby().agg()` | You need flexible aggregation, non-grid output, or you'll keep working with the result as a DataFrame |
| `pd.crosstab()` | You want **counts or proportions** of two categorical variables — quick exploratory tables |
| `pd.pivot_table()` | You want **any aggregation** displayed as a 2-D grid of two categorical variables |

The same question can be answered with any of the three:

In [ ]:
# Three ways to get average survival rate by pclass and sex

# 1. groupby — flat output, needs .unstack() to make it 2-D
r1 = df.groupby(["pclass","sex"])["survived"].mean().unstack("sex").round(2)

# 2. pivot_table — directly 2-D
r2 = pd.pivot_table(df, values="survived", index="pclass", columns="sex", aggfunc="mean").round(2)

print("groupby + unstack:")
print(r1)
print()
print("pivot_table:")
print(r2)

## 4. Correlation — Which Variables Move Together?

You now have survival rates broken down by class and sex. But how do the numeric variables — age, fare, sibsp, parch — relate to each other and to survival?

`.corr()` computes the **Pearson correlation** between every pair of numeric columns. Correlation ranges from -1 (perfect negative relationship) to +1 (perfect positive relationship). Zero means no linear relationship.

In [ ]:
df[["survived", "pclass", "age", "sibsp", "parch", "fare"]].corr().round(2)

A few things to notice:

- **`pclass` and `survived`** (-0.34): a negative correlation means higher class number (3rd class) goes with lower survival — consistent with what the pivot table showed.
- **`fare` and `survived`** (+0.26): higher fares go with higher survival, but this is largely explained by class. First-class passengers paid more *and* survived more — the two variables are tangled.
- **`fare` and `pclass`** (-0.55): a strong negative correlation, confirming that first-class passengers (pclass=1) paid dramatically more.

Correlation captures *linear* association, not causation. Paying a higher fare didn't cause survival — it was a proxy for class, and class was a proxy for proximity to lifeboats and priority in boarding them.

## 5. Finding Outliers with `.nlargest()` and `.nsmallest()`

Correlation tells you about patterns across the whole dataset. Sometimes you want to zoom in on the extremes — who paid the most? Who was oldest?

In [ ]:
# Who paid the highest fares? Did they survive?
df.nlargest(8, "fare")[["name", "pclass", "fare", "age", "survived"]]

In [ ]:
# Who were the youngest passengers?
df.nsmallest(8, "age")[["name", "pclass", "age", "survived"]]

The oldest passengers who paid the highest fares were almost all first-class — and most survived. The youngest passengers are a mix of classes, and their survival is more mixed.

## 6. A Complete Descriptive Summary

Now we have all the tools. Here is a concise data profile of the Titanic dataset — the kind of summary you'd write before a deeper analysis.

In [ ]:
print("=" * 55)
print("TITANIC — DESCRIPTIVE PROFILE")
print("=" * 55)
print(f"\n{len(df):,} passengers | {df['survived'].mean():.0%} survival rate overall")

print("\nSurvival by sex:")
print(df.groupby("sex")["survived"].mean().apply("{:.0%}".format).to_string())

print("\nSurvival by class:")
print(df.groupby("pclass")["survived"].mean().apply("{:.0%}".format).to_string())

print("\nSurvival rate — class × sex:")
print(pd.pivot_table(df, values="survived", index="pclass",
                     columns="sex", aggfunc="mean").round(2).to_string())

print("\nFare (£):")
print(df["fare"].describe().round(2)[["mean","50%","min","max"]].to_string())

print("\nAge (years):")
print(df["age"].describe().round(1)[["mean","50%","min","max"]].to_string())

## Your Turn

1. Use `pd.crosstab()` with `normalize="index"` to compare survival rates across passenger class. Which class had the highest rate?

2. Build a pivot table showing the **median fare** with class as rows and sex as columns. Add margins. Is the gender gap in fares larger in some classes than others?

3. Among the 10 passengers who paid the highest fares, how many survived? Is that rate higher or lower than the overall survival rate?

4. The correlation between `age` and `fare` is small. Does that mean age and class are unrelated? Compute average age by class to find out.

5. Use `pd.crosstab(df["pclass"], df["sex"], normalize="all")` to find what percentage of all passengers fell into each class/sex combination. Which was the largest group? Which was the smallest?

In [ ]:
# Your code here